<a href="https://colab.research.google.com/github/VincenzoDamico/ProteinDynamics/blob/LSTM/LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).



# Library setting



In [8]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import LSTM
from sklearn.preprocessing import StandardScaler
import os
import random

Seeting the seeds:

In [9]:
# fix random seed for reproducibility
def set_all_seeds(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'

set_all_seeds(42)
np.random.seed(42) #it also fix the seed for sklearn

# Inizialization of Input Data

In [10]:
driver_path='/content/drive/MyDrive/ProteinDynamics/'

In [11]:
coordinates_df = pd.read_csv(driver_path+'coordinates.csv', header=None, index_col=False, sep=';')
# header= none: we don't have col names in the first raw
# index_col: we don't have a col for the index
print(coordinates_df.head(), coordinates_df.shape)

# Forces:
forces_df_raw = pd.read_csv(driver_path+'forces.csv', header=None, index_col=False, sep=None, engine='python')
# sep = none means pandas will search to find the separator
forces_df = forces_df_raw[0].str.split('\t', expand=True)
forces_df = forces_df.apply(pd.to_numeric, errors='coerce')
print(forces_df.head(), forces_df.shape)

# velocity:
velocities_df_raw = pd.read_csv(driver_path+'velocity.csv', header=None, index_col=False, sep=None, engine='python')
velocities_df = velocities_df_raw[0].str.split('\t', expand=True)
velocities_df = velocities_df.apply(pd.to_numeric, errors='coerce')
print(velocities_df.head(), velocities_df.shape)

     0        1        2        3        4        5        6        7    \
0  0.000  56.3980  51.9370  52.6880  56.3750  52.0050  52.6170  56.4260   
1  0.002  56.3992  51.9368  52.6877  56.3743  52.0037  52.6162  56.4267   
2  0.004  56.3999  51.9366  52.6871  56.3767  52.0046  52.6161  56.4285   
3  0.006  56.4001  51.9364  52.6861  56.3816  52.0073  52.6167  56.4315   
4  0.008  56.4001  51.9362  52.6848  56.3876  52.0109  52.6179  56.4350   

       8        9    ...      405      406      407      408      409  \
0  51.8530  52.6400  ...  52.1880  56.7460  52.0040  52.3850  56.7650   
1  51.8556  52.6344  ...  52.1872  56.7448  52.0025  52.3849  56.7651   
2  51.8579  52.6307  ...  52.1890  56.7434  52.0011  52.3842  56.7652   
3  51.8594  52.6288  ...  52.1930  56.7417  52.0001  52.3832  56.7652   
4  51.8597  52.6288  ...  52.1982  56.7402  51.9995  52.3824  56.7651   

       410      411      412      413      414  
0  51.9040  52.4540  56.7550  52.1200  52.4210  
1  51.9044  


LSTMs are sensitive to the scale of the input data, specifically when the sigmoid (default) or tanh activation functions are used. It can be a good practice to rescale the data to the range of 0-to-1



In [12]:
coords = coordinates_df.iloc[:, 1:].values
vels   = velocities_df.iloc[:, 1:].values
forces = forces_df.iloc[:, 1:].values
timestamp=coordinates_df.iloc[:,0].values


#RESHAPE TO (timesteps, atoms, xyz)
NUM_ATOMS = 138

coords = coords.reshape(-1,NUM_ATOMS, 3)
vels   = vels.reshape(-1, NUM_ATOMS, 3)
forces = forces.reshape(-1, NUM_ATOMS, 3)

print("\nAfter reshape:")
print("Coordinates:", coords.shape)
print("Velocities :", vels.shape)
print("Forces     :", forces.shape)

#3- Concatenate Features to be [x,y,z,vx,vy,vz,fx,fy,fz]
features = np.concatenate(
    [coords, vels, forces],
    axis=-1
)

print("\nCombined features shape:")
print(features.shape)
N         = len(features)       # 50001
train_end = int(N * 0.70)       # 35000
val_end   = int(N * 0.85)       # 42500

# Fit scaler on training data ONLY — prevents leakage from future frames
train_data = features[:train_end].reshape(-1, 9)   # (35000*138, 9)

scaler = StandardScaler()
scaler.fit(train_data)

def scale_data(data):
    T = data.shape[0]
    data_scaled = scaler.transform(data.reshape(-1, 9))
    return data_scaled.reshape(T, 138, 9).astype('float32')

# features shape = (50001,138,9)

features_scaled = scale_data(features)





After reshape:
Coordinates: (50001, 138, 3)
Velocities : (50001, 138, 3)
Forces     : (50001, 138, 3)

Combined features shape:
(50001, 138, 9)


Adjust the input data to match the LMTS Model standard. The LSTM network expects the input data (X) to be provided with a specific array structure in the form of [samples, time steps, features].


# Coordinate prediction

In this case i will build my feature vector as
my previous 3 positions and the previous force and velocity.

In [ ]:
import pandas as pd
import numpy as np
# we know we're gonna have 5 rows of data
numberOfRows = 5
# create dataframe
df = pd.DataFrame(index=np.arange(0, numberOfRows), columns=('lib', 'qty1', 'qty2') )

# now fill it up row by row
for x in np.arange(0, numberOfRows):
    #loc or iloc both work here since the index is natural numbers
    df.loc[x] = [np.random.randint(-1,1) for n in range(3)]

In [34]:
look_back=3
def feature_def(features_scaled,timestamp,look_back):
  d=pd.DataFrame(index=np.arange(0, features_scaled.shape[0]-look_back),columns=['timestamp','DataForPrediciton'])
  for i in np.arange(0,features_scaled.shape[0]-look_back):
    raw=features_scaled[i:i+look_back]
    d.loc[i]=[timestamp[i],raw]
  return d

print(feature_def(features_scaled,timestamp,look_back).shape)

(49998, 2)
